# Load AlpacaCode-20K Dataset

# Imports

In [ ]:
!pip install unsloth
!pip install --upgrade transformers
!pip install "datasets>=3.4.1,<4.4.0" "trl>=0.18.2,<=0.24.0"
!pip install --upgrade unsloth unsloth-zoo

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.6/62.6 MB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 75.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 418.4/418.4 kB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 60.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 72.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 56.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.8

# Training Setup

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048 # Context Length?
dtype = None        # Auto pick
load_in_4bit = True # Quantize model

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-Coder-0.5B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42
)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.4: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/457M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/266 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/632 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

Unsloth 2026.4.4 patched 24 layers with 24 QKV layers, 24 O layers and 24 MLP layers.


# Dataset Pre-Processing

In [ ]:
from datasets import load_dataset

acds = load_dataset("sahil2801/CodeAlpaca-20k", split="train")

def format_batch(batch_examples):
  instrs = batch_examples["instruction"]
  inputs = batch_examples["input"]
  outputs = batch_examples["output"]

  processed_samples = []

  for instr, input, output in zip(instrs, inputs, outputs):

    user_msg = instr
    if input.strip() != "":
      user_msg = instr + "\n\n" + input

    roled_samples = [
        {
            "role": "system",
            "content": "You are a coding assistant."
        },
        {
            "role": "user",
            "content": user_msg
        },
        {
            "role": "assistant",
            "content": output
        }
    ]

    sample = tokenizer.apply_chat_template(
        roled_samples,
        tokenize=False,
        add_generation_prompt=False
    )

    processed_samples.append(sample)

  return {"text": processed_samples}

acds = acds.map(format_batch, batched=True)


print(acds[0]["text"])

README.md:   0%|          | 0.00/147 [00:00<?, ?B/s]

code_alpaca_20k.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/20022 [00:00<?, ? examples/s]

Map:   0%|          | 0/20022 [00:00<?, ? examples/s]

<|im_start|>system
You are a coding assistant.<|im_end|>
<|im_start|>user
Create an array of length 5 which contains all even numbers between 1 and 10.<|im_end|>
<|im_start|>assistant
arr = [2, 4, 6, 8, 10]<|im_end|>



# Fine-Tuning

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments, trainer_utils

from google.colab import drive
import os
from time import sleep

import shutil



drive.mount('/content/drive')

# Find and copy checkpoint locally
def get_resume_checkpoint(drive_checkpoint_dir, local_dir="/content/checkpoints"):
    last_checkpoint = trainer_utils.get_last_checkpoint(drive_checkpoint_dir)

    if last_checkpoint is None:
        print("No checkpoint found - starting fresh")
        return



    print(f"Found checkpoint: {last_checkpoint}")
    local_checkpoint = os.path.join(local_dir, os.path.basename(last_checkpoint))

    if not os.path.exists(local_checkpoint):
        print("Copying checkpoint to local storage...")
        os.makedirs(local_dir, exist_ok=True)
        shutil.copytree(last_checkpoint, local_checkpoint)
        print("Copy complete!")

    return local_checkpoint

CHECKPOINT_DIR = "/content/drive/MyDrive/fine_tuning/checkpoint_weights"

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = acds,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 3,
        max_steps = -1,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        output_dir = CHECKPOINT_DIR,
        save_strategy = "steps",
        save_steps = 100,
        save_total_limit = 3,
    )
)

checkpoint = get_resume_checkpoint(CHECKPOINT_DIR)
trainer.train(resume_from_checkpoint = checkpoint)

model.save_pretrained("/content/drive/MyDrive/fine_tuning/weights")
tokenizer.save_pretrained("/content/drive/MyDrive/fine_tuning/weights")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found checkpoint: /content/drive/MyDrive/fine_tuning/checkpoint_weights/checkpoint-3000
Copying checkpoint to local storage...
Copy complete!


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 20,022 | Num Epochs = 3 | Total steps = 7,509
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 8,798,208 of 502,830,976 (1.75% trained)


Step,Training Loss
3001,0.806998
3002,0.529248
3003,0.459046
3004,0.566908
3005,0.889217
3006,0.535821
3007,0.691648
3008,0.568601
3009,0.761055
3010,0.753864


('/content/drive/MyDrive/fine_tuning/weights/tokenizer_config.json',
 '/content/drive/MyDrive/fine_tuning/weights/chat_template.jinja',
 '/content/drive/MyDrive/fine_tuning/weights/tokenizer.json')